# 🎓 Face Recognition Training (FaceNet + SVM)
ระบบเทรนใบหน้าระดับสูง แม่นยำ >95%
1. **MTCNN**: ตรวจจับใบหน้าแม่นยำทุกมุม
2. **InceptionResnetV1 (FaceNet)**: สกัดใบหน้าเป็นตัวเลข 512 มิติ
3. **SVM Classifier**: จำแนกบุคคล
4. **Add New Person**: เพิ่มคนใหม่ผ่านกล้องได้ทันที


In [ ]:
# 1. เชื่อมต่อ Google Drive (สำหรับ Google Colab)
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ เชื่อมต่อ Google Drive สำเร็จ')
    IS_COLAB = True
except ImportError:
    print('ℹ️ รันใน Local Environment')
    IS_COLAB = False


In [ ]:
!pip install facenet-pytorch --no-deps
!pip install scikit-learn joblib -q

import torch
import numpy as np
import cv2
import json
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from facenet_pytorch import MTCNN, InceptionResnetV1
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from PIL import Image

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'💻 ใช้หน่วยประมวลผล: {device}')

# สร้าง MTCNN และ ResNet
print('กำลังโหลดโมเดล FaceNet...')
mtcnn = MTCNN(image_size=160, margin=20, keep_all=False, select_largest=True, post_process=True, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)
print('✅ โหลดโมเดลสำเร็จ!')


In [ ]:
# 3. ค้นหาโฟลเดอร์ Dataset
possible_paths = [
    'dataset', './dataset', '../dataset',
    '/content/drive/MyDrive/Project_Ai/dataset',
    os.path.join(os.getcwd(), 'dataset')
]

DATASET_PATH = None
for p in possible_paths:
    if os.path.exists(p):
        DATASET_PATH = p
        break

if DATASET_PATH is None:
    DATASET_PATH = '/content/drive/MyDrive/Project_Ai/dataset'
    os.makedirs(DATASET_PATH, exist_ok=True)
    print(f'สร้างโฟลเดอร์ใหม่ที่: {DATASET_PATH}')

print(f'📂 โฟลเดอร์ Dataset: {DATASET_PATH}')


In [ ]:
# 4. ฟังก์ชันเพิ่มคนใหม่เข้า Dataset (Live Webcam Colab)
from IPython.display import display, Javascript, clear_output
from base64 import b64decode
import time

def capture_new_person_colab(person_name, num_images=10):
    save_dir = os.path.join(DATASET_PATH, person_name)
    os.makedirs(save_dir, exist_ok=True)
    
    js = Javascript("""
        async function takePhoto() {
            const div = document.createElement('div');
            const video = document.createElement('video');
            video.style.display = 'block';
            video.style.maxWidth = '480px';
            div.appendChild(video);
            document.body.appendChild(div);
            
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            video.srcObject = stream;
            await video.play();
            
            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
            
            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            
            window.captureFrame = function() {
                canvas.getContext('2d').drawImage(video, 0, 0);
                return canvas.toDataURL('image/jpeg', 0.9);
            };
            
            window.stopStream = function() {
                stream.getVideoTracks()[0].stop();
                div.remove();
            };
        }
    """)
    display(js)
    try:
        from google.colab.output import eval_js
        eval_js('takePhoto()')
        print(f'📸 กรุณาหันหน้ามุมต่างๆ (เก็บ {num_images} ภาพ)...')
        time.sleep(2)
        
        saved_count = 0
        for i in range(num_images):
            data = eval_js('captureFrame()')
            binary = b64decode(data.split(',')[1])
            img_arr = np.frombuffer(binary, dtype=np.uint8)
            img_cv = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)
            
            # ตรวจจับใบหน้าก่อนเซฟ
            img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(img_rgb)
            face = mtcnn(pil_img)
            
            if face is not None:
                img_path = os.path.join(save_dir, f'img_{int(time.time())}_{i}.jpg')
                cv2.imwrite(img_path, img_cv)
                saved_count += 1
                print(f'  ✓ บันทึกภาพที่ {saved_count}/{num_images}')
            else:
                print('  ❌ ไม่พบใบหน้า กรุณาขยับใหม่')
            time.sleep(0.5)
            
        eval_js('stopStream()')
        print(f'✅ เก็บภาพ {person_name} เสร็จสิ้น {saved_count} ภาพ!')
    except Exception as e:
        print(f'⚠️ ไม่สามารถเปิดกล้องบน Colab ได้: {e}')

# วิธีใช้งาน (ปลดคอมเมนต์เพื่อใช้งาน):
# if IS_COLAB:
#     capture_new_person_colab('New_Person_Name', num_images=10)


In [ ]:
from torchvision import transforms

fallback_transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

def extract_embeddings(dataset_path):
    X = []
    y = []
    
    person_names = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])
    print(f'พบ {len(person_names)} บุคคล: {person_names}')
    
    for person in person_names:
        person_dir = os.path.join(dataset_path, person)
        image_files = [f for f in os.listdir(person_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        
        count = 0
        for img_name in image_files:
            img_path = os.path.join(person_dir, img_name)
            img = Image.open(img_path).convert('RGB')
            
            # ตรวจจับและครอบใบหน้า
            face_tensor = mtcnn(img)
            if face_tensor is not None:
                X.append(face_tensor)
                y.append(person)
                count += 1
            else:
                # Fallback: กรณีเป็นรูปที่ถูก Crop หน้ามาแล้ว MTCNN อาจจะหาไม่เจอ ให้แปลงตรงๆ เลย
                face_tensor = fallback_transform(img)
                X.append(face_tensor)
                y.append(person)
                count += 1
                
        print(f'[{person}] สกัดได้ {count}/{len(image_files)} ภาพ')
        
    if len(X) == 0:
        return np.array([]), np.array([])
        
    X_tensor = torch.stack(X).to(device)
    
    batch_size = 32
    embeddings = []
    print(f'\nกำลังแปลงรูปเป็นตัวเลข 512 มิติ (จำนวน {len(X)} ภาพ)...')
    
    resnet.eval()
    with torch.no_grad():
        for i in range(0, len(X_tensor), batch_size):
            batch = X_tensor[i:i+batch_size]
            emb = resnet(batch).cpu().numpy()
            embeddings.append(emb)
            
    X_embed = np.vstack(embeddings)
    y = np.array(y)
    print('✅ การสกัด Feature เสร็จสมบูรณ์!')
    return X_embed, y

X_embed, y = extract_embeddings(DATASET_PATH)



In [ ]:
# 6. เทรนโมเดล SVM Classifier
if len(X_embed) > 0:
    # แปลงชื่อคนเป็นตัวเลข (Label Encoding)
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    # แบ่ง Train / Test
    X_train, X_test, y_train, y_test = train_test_split(X_embed, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
    
    print(f'📊 ข้อมูล Train: {len(X_train)} | Test: {len(X_test)}')
    
    # เทรน SVM (ใช้ probability=True เพื่อให้บอก % ความมั่นใจได้)
    print('กำลังเทรน SVM Classifier...')
    clf = SVC(kernel='linear', probability=True)
    clf.fit(X_train, y_train)
    print('✅ เทรนเสร็จสมบูรณ์!')
else:
    print('❌ ไม่มีข้อมูลสำหรับเทรน')


In [ ]:
# 7. ประเมินผลความแม่นยำ
if len(X_embed) > 0:
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    print('='*50)
    print(f'🎯 Accuracy บน Test Set: {acc*100:.2f}%')
    print('='*50)
    
    target_names = le.classes_
    print('\n📑 Classification Report:')
    print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))
    
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()


In [ ]:
# 8. บันทึกโมเดล
if len(X_embed) > 0:
    save_dir = os.path.dirname(DATASET_PATH)
    if not os.path.exists(save_dir):
        save_dir = '.'
        
    clf_path = os.path.join(save_dir, 'facenet_svm_model.pkl')
    le_path = os.path.join(save_dir, 'label_encoder.pkl')
    
    joblib.dump(clf, clf_path)
    joblib.dump(le, le_path)
    
    print(f'💾 บันทึกโมเดล SVM ที่: {clf_path}')
    print(f'💾 บันทึก Label Encoder ที่: {le_path}')
